🏦 Bank Marketing Deposit Prediction — ML Pipeline
Nama: Ivan Rahadian
Username Dicoding: rahadianivan09
Tentang Proyek
Notebook ini membangun pipeline ML end-to-end menggunakan TFX (TensorFlow Extended) untuk memprediksi apakah seorang nasabah bank akan berlangganan deposito berjangka atau tidak.
Dataset
Bank Marketing Dataset — UCI via Kaggle

11.162 baris, 16 fitur demografis dan riwayat kampanye marketing
Label: deposit — 1 (yes) atau 0 (no)

Tujuan
Mengklasifikasikan nasabah potensial secara otomatis sehingga kampanye marketing bank menjadi lebih efisien dan tepat sasaran.
Alur Pipeline
ExampleGen → StatisticsGen → SchemaGen → ExampleValidator
→ Transform → Tuner → Trainer → Resolver → Evaluator → Pusher

🔧 Cell 1 — Verifikasi Environment
Sebelum menjalankan pipeline, kita memastikan semua dependency utama sudah tersedia dan versinya kompatibel.

- PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python diperlukan untuk menghindari konflik antara protobuf versi C++ dan Python pada beberapa environment.
- TF_CPP_MIN_LOG_LEVEL=3 menyembunyikan log verbose TensorFlow agar output lebih bersih.
- Verifikasi versi TFX dan TensorFlow dilakukan di sini untuk memastikan kompatibilitas komponen pipeline.

In [1]:
# ── CELL 1: Verifikasi Environment ───────────────
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print(os.environ.get("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "NOT SET"))

import tfx
print("TFX version:", tfx.__version__)
import tensorflow as tf
print("TF version:", tf.__version__)

import warnings
warnings.filterwarnings("ignore")

python
TFX version: 1.14.0


/workspaces/rahadianivan09-mlpipeline/tfx-env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


TF version: 2.13.1


## 📦 Cell 2 — Import Library

Import seluruh library yang dibutuhkan sepanjang pipeline:

| Modul | Kegunaan |
|---|---|
| `tensorflow` / `tfx` | Framework ML dan pipeline orchestration |
| `tensorflow_model_analysis` | Evaluasi model dengan TFMA |
| `CsvExampleGen`, `StatisticsGen`, dst. | Komponen-komponen TFX pipeline |
| `InteractiveContext` | Menjalankan pipeline interaktif di notebook |

Semua komponen pipeline — dari **ExampleGen** hingga **Pusher** — di-import di cell ini.

In [2]:
# ── CELL 2: Import Library ───────────────────────
import os
import json
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_model_analysis as tfma
import tfx

from tfx.components import (
    CsvExampleGen, StatisticsGen, SchemaGen,
    ExampleValidator, Transform, Trainer,
    Evaluator, Pusher, Tuner
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.orchestration.experimental.interactive.interactive_context import (
    InteractiveContext,
)

print("Semua library berhasil diimport!")

Semua library berhasil diimport!


📁 Cell 3 — Setup Path
Mendefinisikan semua path yang digunakan selama pipeline berjalan.

rahadianivan09-pipeline/
├── data/                  ← dataset bank.csv
├── modules/               ← transform & trainer module
└── rahadianivan09-pipeline/
    ├── metadata/          ← SQLite metadata store
    └── serving_model/     ← model yang sudah di-push

Semua direktori dibuat otomatis dengan os.makedirs(..., exist_ok=True) sehingga tidak error meski dijalankan berulang kali.

In [3]:
# ── CELL 3: Setup Path ───────────────────────────
PROJECT_ROOT = os.getcwd()
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
TRANSFORM_MODULE = os.path.join(PROJECT_ROOT, "modules", "rahadianivan09_transform.py")
TRAINER_MODULE = os.path.join(PROJECT_ROOT, "modules", "rahadianivan09_trainer.py")
PIPELINE_ROOT = os.path.join(PROJECT_ROOT, "rahadianivan09-pipeline")
METADATA_PATH = os.path.join(PIPELINE_ROOT, "metadata", "metadata.db")
SERVING_MODEL = os.path.join(PIPELINE_ROOT, "serving_model")

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(PIPELINE_ROOT, exist_ok=True)
os.makedirs(os.path.dirname(METADATA_PATH), exist_ok=True)
os.makedirs(SERVING_MODEL, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data         : {DATA_ROOT}")
print(f"Pipeline root: {PIPELINE_ROOT}")
print(f"Serving model: {SERVING_MODEL}")

Project root : /workspaces/rahadianivan09-mlpipeline
Data         : /workspaces/rahadianivan09-mlpipeline/data
Pipeline root: /workspaces/rahadianivan09-mlpipeline/rahadianivan09-pipeline
Serving model: /workspaces/rahadianivan09-mlpipeline/rahadianivan09-pipeline/serving_model


📊 Cell 4 — Load Dataset
Memuat dataset Bank Marketing UCI dari file data/bank.csv.

- Total baris: 11.162 sampel
- Fitur: 16 kolom (demografis nasabah + hasil kampanye marketing)
- Label: deposit — apakah nasabah berlangganan deposito berjangka (yes/no)

Label deposit dikonversi dari string menjadi integer biner (1 = yes, 0 = no) agar kompatibel dengan model klasifikasi. 
Konversi hanya dilakukan sekali (dicek dulu dtype-nya).

In [4]:
# ── CELL 4: Load Dataset ─────────────────────────
df = pd.read_csv(os.path.join(DATA_ROOT, "bank.csv"))

# Konversi hanya kalau masih berupa string yes/n
if df['deposit'].dtype == object:
    df['deposit'] = (df['deposit'] == 'yes').astype(int)
    df.to_csv(os.path.join(DATA_ROOT, "bank.csv"), index=False)

print(f"Total data      : {len(df)}")
print(f"Deposit (1=yes) : {df['deposit'].sum()}")
print(f"No deposit (0)  : {len(df) - df['deposit'].sum()}")
df.head()

Total data      : 11162
Deposit (1=yes) : 5289
No deposit (0)  : 5873


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,1
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,1
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,1
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,1
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,1


⚙️ Cell 5 — InteractiveContext
InteractiveContext adalah interface TFX untuk menjalankan pipeline secara interaktif di Jupyter Notebook.

- Pipeline name: bank-deposit
- Pipeline root: rahadianivan09-pipeline/
- Metadata store: SQLite di rahadianivan09-pipeline/metadata/metadata.db

Metadata store menyimpan lineage setiap artefak yang dihasilkan komponen pipeline, sehingga hasil setiap langkah bisa di-cache dan tidak perlu dijalankan ulang.

In [5]:
# ── CELL 5: InteractiveContext ───────────────────
context = InteractiveContext(
    pipeline_name="bank-deposit",
    pipeline_root=PIPELINE_ROOT,
    metadata_connection_config=tfx.orchestration.metadata.sqlite_metadata_connection_config(
        METADATA_PATH
    ),
)
print("InteractiveContext berhasil dibuat!")

InteractiveContext berhasil dibuat!


📥 Cell 6 — ExampleGen
ExampleGen adalah komponen pertama dan entry point pipeline TFX. 
Komponen ini membaca data mentah dari data/bank.csv dan mengonversinya ke format TFRecord yang dioptimalkan untuk TensorFlow.
Secara default, data dibagi menjadi:

- Train split: 80% dari dataset (~8.929 baris)
- Eval split: 20% dari dataset (~2.233 baris)

Output artefak disimpan di pipeline root dan siap digunakan oleh komponen selanjutnya.

In [6]:
# ── CELL 6: ExampleGen ───────────────────────────
example_gen = CsvExampleGen(input_base=DATA_ROOT)
context.run(example_gen)
print("ExampleGen selesai!")

ExampleGen selesai!


📈 Cell 7 — StatisticsGen
StatisticsGen menghitung statistik deskriptif dari setiap fitur di dataset (train & eval split).
Statistik yang dihasilkan meliputi:

- Distribusi nilai per fitur (histogram)
- Mean, median, standar deviasi (fitur numerik)
- Frekuensi nilai unik (fitur kategorik)
- Missing value count

Output statistik divisualisasikan langsung di notebook menggunakan context.show() via TFDV (TensorFlow Data Validation).

In [7]:
# ── CELL 7: StatisticsGen ────────────────────────
statistics_gen = StatisticsGen(
    examples=example_gen.outputs["examples"]
)
context.run(statistics_gen)
context.show(statistics_gen.outputs["statistics"])

🗂️ Cell 8 — SchemaGen
SchemaGen secara otomatis meng-infer schema dataset berdasarkan statistik dari StatisticsGen.
Schema mendefinisikan:

- Tipe data setiap fitur (INT, FLOAT, BYTES)
- Domain nilai yang valid (min/max untuk numerik, vocabulary untuk kategorik)
- Fitur mana yang required/optional

Schema ini menjadi "kontrak data" yang digunakan oleh ExampleValidator dan Transform untuk memvalidasi dan memproses data secara konsisten.

In [8]:
# ── CELL 8: SchemaGen ────────────────────────────
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs["statistics"]
)
context.run(schema_gen)
context.show(schema_gen.outputs["schema"])

,Type,Presence,Valency,Domain
Feature name,,,,
'age',INT,required,,-
'balance',INT,required,,-
'campaign',INT,required,,-
'contact',STRING,required,,'contact'
'day',INT,required,,-
'default',STRING,required,,'default'
'deposit',INT,required,,-
'duration',INT,required,,-
'education',STRING,required,,'education'


,Values
Domain,
'contact',"'cellular', 'telephone', 'unknown'"
'default',"'no', 'yes'"
'education',"'primary', 'secondary', 'tertiary', 'unknown'"
'housing',"'no', 'yes'"
'job',"'admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown'"
'loan',"'no', 'yes'"
'marital',"'divorced', 'married', 'single'"
'month',"'apr', 'aug', 'dec', 'feb', 'jan', 'jul', 'jun', 'mar', 'may', 'nov', 'oct', 'sep'"
'poutcome',"'failure', 'other', 'success', 'unknown'"


✅ Cell 9 — ExampleValidator
ExampleValidator memvalidasi setiap contoh data terhadap schema yang sudah di-infer oleh SchemaGen.
Komponen ini mendeteksi anomali seperti:
- Missing values melebihi threshold
- Nilai di luar domain yang ditentukan schema
- Tipe data tidak sesuai

Output berupa laporan anomali yang ditampilkan via context.show(). Jika tidak ada anomali, pipeline aman dilanjutkan ke tahap transformasi.

In [9]:
# ── CELL 9: ExampleValidator ────────────────────
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"],
)
context.run(example_validator)
context.show(example_validator.outputs["anomalies"])

🔄 Cell 10 — Transform
Transform melakukan feature engineering dan preprocessing menggunakan modul kustom rahadianivan09_transform.py.
Preprocessing yang dilakukan:
- One-hot encoding untuk fitur kategorik (job, marital, education, dll.)
- Normalisasi untuk fitur numerik (age, balance, duration, dll.)
- Vocabulary lookup yang konsisten antara training dan serving

Output komponen ini adalah transformed_examples (data siap latih) dan transform_graph (graf transformasi yang di-embed ke serving model).

In [10]:
# ── CELL 10: Transform ───────────────────────────
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=os.path.abspath(TRANSFORM_MODULE),
)
context.run(transform)
print("Transform selesai!")

running bdist_wheel
running build
running build_py
creating build/lib
copying rahadianivan09_transform.py -> build/lib
copying rahadianivan09_trainer.py -> build/lib
installing to /tmp/tmp8yyzrkl0
running install
running install_lib
copying build/lib/rahadianivan09_transform.py -> /tmp/tmp8yyzrkl0/.
copying build/lib/rahadianivan09_trainer.py -> /tmp/tmp8yyzrkl0/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmp8yyzrkl0/./tfx_user_code_Transform-0.0+16ad65ed7a80e0ff3dbf7191f5a3234d64cc6f42267b6

/workspaces/rahadianivan09-mlpipeline/tfx-env/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


Processing ./rahadianivan09-pipeline/_wheels/tfx_user_code_transform-0.0+16ad65ed7a80e0ff3dbf7191f5a3234d64cc6f42267b66c6dac7829d15764caf-py3-none-any.whl



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Processing ./rahadianivan09-pipeline/_wheels/tfx_user_code_transform-0.0+16ad65ed7a80e0ff3dbf7191f5a3234d64cc6f42267b66c6dac7829d15764caf-py3-none-any.whl


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


Transform selesai!


🔍 Cell 11 — Tuner ✅
Tuner melakukan hyperparameter search otomatis menggunakan Keras Tuner yang didefinisikan di rahadianivan09_trainer.py.
Hyperparameter yang di-tune: jumlah unit per layer, learning rate, dropout rate, dan arsitektur layer.
Konfigurasi pencarian:

- Train steps: 200
- Eval steps: 100

Best hyperparameter yang ditemukan otomatis diteruskan ke Trainer sebagai input, sehingga model dilatih dengan konfigurasi optimal.

In [11]:
# ── CELL 11: Tuner ───────────────────────────────
tuner = Tuner(
    module_file=os.path.abspath(TRAINER_MODULE),
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=200),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=100),
)
context.run(tuner)
print("Tuner selesai!")

Trial 3 Complete [00h 00m 07s]
val_auc: 0.7588983178138733

Best val_auc So Far: 0.7588983178138733
Total elapsed time: 00h 00m 22s
INFO:tensorflow:Oracle triggered exit


INFO:tensorflow:Oracle triggered exit


Results summary
Results in /workspaces/rahadianivan09-mlpipeline/rahadianivan09-pipeline/.temp/26/bank_tuning
Showing 10 best trials
Objective(name="val_auc", direction="max")

Trial 2 summary
Hyperparameters:
units_1: 256
units_2: 96
units_3: 16
dropout: 0.2
learning_rate: 0.01
Score: 0.7588983178138733

Trial 1 summary
Hyperparameters:
units_1: 192
units_2: 64
units_3: 16
dropout: 0.2
learning_rate: 0.0001
Score: 0.7063679695129395

Trial 0 summary
Hyperparameters:
units_1: 64
units_2: 32
units_3: 32
dropout: 0.30000000000000004
learning_rate: 0.0001
Score: 0.6538282632827759
Tuner selesai!


🏋️ Cell 12 — Trainer
Trainer melatih model klasifikasi deposit menggunakan best hyperparameter dari Tuner.
Input yang digunakan: transformed_examples, transform_graph, best_hyperparameters, dan schema.
Konfigurasi pelatihan:

- Train steps: 100
- Eval steps: 50

Model yang dihasilkan disimpan sebagai artefak TFX dan siap dievaluasi oleh Evaluator.

In [12]:
# ── CELL 12: Trainer ─────────────────────────────
trainer = Trainer(
    module_file=os.path.abspath(TRAINER_MODULE),
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    hyperparameters=tuner.outputs["best_hyperparameters"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=100),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=50),
)
context.run(trainer)
print("Trainer selesai!")

/workspaces/rahadianivan09-mlpipeline/tfx-env/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


running bdist_wheel
running build
running build_py
creating build/lib
copying rahadianivan09_transform.py -> build/lib
copying rahadianivan09_trainer.py -> build/lib
installing to /tmp/tmp6p1dq057
running install
running install_lib
copying build/lib/rahadianivan09_transform.py -> /tmp/tmp6p1dq057/.
copying build/lib/rahadianivan09_trainer.py -> /tmp/tmp6p1dq057/.
running install_egg_info
running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /tmp/tmp6p1dq057/./tfx_user_code_Trainer-0.0+16ad65ed7a80e0ff3dbf7191f5a3234d64cc6f42267b66c6dac7829d15764ca


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 job_xf (InputLayer)         [(None, 1)]                  0         []                            
                                                                                                  
 marital_xf (InputLayer)     [(None, 1)]                  0         []                            
                                                                                                  
 education_xf (InputLayer)   [(None, 1)]                  0         []                            
                                                                                                  
 default_xf (InputLayer)     [(None, 1)]                  0         []                            
                                                                                            

INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: /workspaces/rahadianivan09-mlpipeline/rahadianivan09-pipeline/Trainer/model/27/Format-Serving/assets


INFO:tensorflow:Assets written to: /workspaces/rahadianivan09-mlpipeline/rahadianivan09-pipeline/Trainer/model/27/Format-Serving/assets


Model berhasil disimpan ke: /workspaces/rahadianivan09-mlpipeline/rahadianivan09-pipeline/Trainer/model/27/Format-Serving
Trainer selesai!


🔗 Cell 13 — Resolver
Resolver dengan strategi LatestBlessedModelStrategy mencari model terbaru yang sudah di-bless dari metadata store.

Fungsi utamanya:
- Menjadi baseline model untuk perbandingan di Evaluator
- Memastikan model baru harus lebih baik dari model produksi sebelumnya sebelum di-push

Jika belum ada model blessed sebelumnya, Evaluator akan menjalankan evaluasi absolut tanpa perbandingan.

In [13]:
# ── CELL 13: Resolver ────────────────────────────
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
)
context.run(model_resolver)
print("Resolver selesai!")

Resolver selesai!


📐 Cell 14 — Evaluator
Evaluator mengevaluasi performa model menggunakan TFMA dengan threshold yang sudah ditentukan.

| Metrik | Threshold | Hasil |
|--------|-----------|-------|
| BinaryAccuracy | ≥ 0.50 | **0.6675** ✅ |
| AUC | ≥ 0.70 | **0.7419** ✅ |

Metrik tambahan yang dihitung: Precision, Recall, dan ExampleCount. Model mendapat status BLESSED karena semua threshold terpenuhi.

In [ ]:
# ── CELL 14: Evaluator ───────────────────
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            label_key="deposit",
            signature_name="serving_default",
        )
    ],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(
                    class_name="BinaryAccuracy",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.50}
                        )
                    )
                ),
                tfma.MetricConfig(
                    class_name="AUC",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.70}
                        )
                    )
                ),
                tfma.MetricConfig(class_name="Precision"),
                tfma.MetricConfig(class_name="Recall"),
                tfma.MetricConfig(class_name="ExampleCount"),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config,
)
context.run(evaluator)
context.show(evaluator.outputs["evaluation"])
print("Evaluator selesai!")

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Overall', 'metrics':…

Evaluator selesai!


🚀 Cell 15 — Pusher
Pusher men-deploy model ke direktori serving jika dan hanya jika model mendapat status BLESSED dari Evaluator.

- Destination: rahadianivan09-pipeline/serving_model/
- Model disimpan dalam format SavedModel yang kompatibel dengan TensorFlow Serving

Untuk bonus, model yang di-push ini selanjutnya di-serve menggunakan Docker + TensorFlow Serving.

In [15]:
# ── CELL 15: Pusher ──────────────────────────────
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL
        )
    ),
)
context.run(pusher)
print(f"Pusher selesai! Model tersimpan di: {SERVING_MODEL}")

Pusher selesai! Model tersimpan di: /workspaces/rahadianivan09-mlpipeline/rahadianivan09-pipeline/serving_model


🗂️ Cell 16 — Verifikasi Hasil Pusher
Cell terakhir memverifikasi bahwa model berhasil di-push dengan menampilkan struktur direktori serving_model/.
Struktur yang diharapkan:
serving_model/
└── <timestamp>/
    ├── saved_model.pb
    ├── fingerprint.pb
    └── variables/
        ├── variables.index
        └── variables.data-00000-of-00001
Adanya direktori bertimestamp mengonfirmasi model telah berhasil di-export dalam format SavedModel dan siap di-serve via TensorFlow Serving atau Docker.

In [16]:
# ── CELL 16: Cek Hasil Pusher ────────────────────
for root, dirs, files in os.walk(SERVING_MODEL):
    level = root.replace(SERVING_MODEL, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

serving_model/
  1780374645/
    saved_model.pb
    fingerprint.pb
    keras_metadata.pb
    variables/
      variables.index
      variables.data-00000-of-00001
    assets/
      job_vocab
      default_vocab
      marital_vocab
      education_vocab
      contact_vocab
      housing_vocab
      loan_vocab
      month_vocab
